In [0]:
%python
# dml/02a_carga_stg_cvm_informe_complemento.ipynb
# MAGIC %pip install requests pandas numpy # Garante as dependencias no compute Serverless

# %%
import requests
import zipfile
import io
import pandas as pd
import numpy as np
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# %%
# 1. Recupera os de-paras de CNPJ e Ticker na tabela dim_fundo_imobiliario
catalogo = "product_dev"
schema = "financas"
tabela_dim = "dim_fundo_imobiliario"

print(f"Buscando CNPJs ativos mapeados na tabela {catalogo}.{schema}.{tabela_dim}...")

# Força o fuso horário de Brasília para garantir consistência de datas
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# Lê apenas fundos que possuem CNPJ mapeados na dimensão
df_dim_ativos = spark.sql(f"""
    SELECT ticker, cnpj 
    FROM {catalogo}.{schema}.{tabela_dim} 
    WHERE cnpj IS NOT NULL
""")

print(f"Total de FIIs ativos com CNPJ para processamento: {df_dim_ativos.count()}")

# %%
# 2. Faz o download do ZIP de 2026 diretamente da CVM
url_zip = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_2026.zip"
headers = {'User-Agent': 'Mozilla/5.0'}

print(f"Baixando pacote anual de Informes Mensais da CVM (isto pode levar de 15s a 30s)...")
response = requests.get(url_zip, headers=headers, timeout=30)
response.raise_for_status()
print("Download do ZIP concluído com sucesso!")

# %%
# 3. Abre o ZIP e extrai o arquivo Complementar da CVM (Tratamento Defensivo)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
arquivo_complemento = "inf_mensal_fii_complemento_2026.csv"

print(f"Lendo e parseando o arquivo contabil {arquivo_complemento} em Pandas...")

with zip_file.open(arquivo_complemento) as f:
    # Lógica defensiva para ler o arquivo mesmo com eventuais linhas desformatadas
    df_pd_complemento = pd.read_csv(
        f, 
        sep=';', 
        encoding='ISO-8859-1', 
        on_bad_lines='skip'
    )

print(f"Total de registros contabeis brutos lidos: {len(df_pd_complemento)}")

# %%
# 4. Transforma para Spark DataFrame e executa a Regra Contabil (Último Mês/Versão)
print("Processando dados contabeis de cotas e dividendos no Spark...")

# Converte de Pandas para Spark de forma nativa e força strings temporárias para limpeza
df_spark_raw = spark.createDataFrame(df_pd_complemento.astype(str))

# Janela contábil particionada por CNPJ e ordenada pelo mês/versão mais recentes
janela_competencia = Window.partitionBy("CNPJ_Fundo_Classe").orderBy(
    F.col("Data_Referencia").desc(), 
    F.col("Versao").desc()
)

df_complemento_filtrado = (
    df_spark_raw
    # Limpa as formatações do CNPJ CVM para cruzamento
    .withColumn("cnpj_limpo", F.regexp_replace(F.col("CNPJ_Fundo_Classe"), r"[\./-]", ""))
    
    # Aplica o Row Number na janela para pegar apenas a competência e versão mais recentes de cada fundo
    .withColumn("rn", F.row_number().over(janela_competencia))
    .filter(F.col("rn") == 1)
    
    # Tratamento defensivo de conversões numéricas (BLINDADO contra overflows contabeis usando try_cast no tipo final)
    .withColumn("valor_pat_cota", F.coalesce(F.col("Valor_Patrimonial_Cotas").cast("double"), F.lit(0.0)))
    .withColumn("pat_liquido", F.coalesce(F.col("Patrimonio_Liquido").cast("double"), F.lit(0.0)))
    # Forçamos o try_cast direto no Spark para os tipos exatos antes da gravação fisica
    .withColumn("cotas_emit", F.coalesce(F.expr("try_cast(Cotas_Emitidas as bigint)"), F.lit(0)))
    .withColumn("num_cotistas", F.coalesce(F.expr("try_cast(Total_Numero_Cotistas as int)"), F.lit(0)))
    
    .withColumn("taxa_adm", F.coalesce(F.col("Percentual_Despesas_Taxa_Administracao").cast("double"), F.lit(0.0)))
    .withColumn("div_yield_mes", F.coalesce(F.col("Percentual_Dividend_Yield_Mes").cast("double"), F.lit(0.0)))
    .withColumn("amort_mes", F.coalesce(F.col("Percentual_Amortizacao_Cotas_Mes").cast("double"), F.lit(0.0)))
)

# %%
# 5. Junta os dados contabeis CVM com os Tickers ativos da nossa Dimensão de FIIs
print("Associando dados contabeis oficiais da CVM com os Tickers ativos da carteira...")

df_complemento_final = (
    df_complemento_filtrado.alias("cvm")
    .join(
        df_dim_ativos.alias("dim"),
        F.col("cvm.cnpj_limpo") == F.col("dim.cnpj"),
        "inner" # INNER JOIN garante que só vamos salvar o que temos na dimensão!
    )
    .select(
        F.col("dim.ticker").alias("ticker"),
        F.col("cvm.Data_Referencia").cast("date").alias("data_referencia"),
        F.col("cvm.cnpj_limpo").alias("cnpj"),
        F.col("cvm.valor_pat_cota").alias("valor_patrimonial_cota"),
        F.col("cvm.pat_liquido").alias("patrimonio_liquido"),
        # Cast explícito e seguro de saída
        F.col("cvm.cotas_emit").cast("bigint").alias("cotas_emitidas"),
        F.col("cvm.num_cotistas").cast("int").alias("total_numero_cotistas"),
        F.col("cvm.taxa_adm").alias("percentual_taxa_administracao"),
        F.col("cvm.div_yield_mes").alias("percentual_dividend_yield_mes"),
        F.col("cvm.amort_mes").alias("percentual_amortizacao_mes"),
        F.from_utc_timestamp(F.current_timestamp(), "America/Sao_Paulo").alias("data_carga")
    )
)

# Cria a view temporária no Spark
df_complemento_final.createOrReplaceTempView("temp_cvm_complemento_consolidado")

# %%
# 6. Execução do INSERT OVERWRITE dinâmico na tabela de Staging de Complementos CVM
tabela_destino = "stg_cvm_informe_complemento"

qry_insert_stg = f"""
  INSERT OVERWRITE {catalogo}.{schema}.{tabela_destino}
  SELECT 
    ticker,
    data_referencia,
    cnpj,
    valor_patrimonial_cota,
    patrimonio_liquido,
    cotas_emitidas,
    total_numero_cotistas,
    percentual_taxa_administracao,
    percentual_dividend_yield_mes,
    percentual_amortizacao_mes,
    data_carga
  FROM temp_cvm_complemento_consolidado
"""

print(f"Gravando dados consolidados de performance em: {catalogo}.{schema}.{tabela_destino}...")

# Executa a gravação atômica
spark.sql(qry_insert_stg)

print("✅ Carga das tabelas contabeis Silver de Complementos CVM concluída com SUCESSO!")